In [61]:
!pip install pandas scikit-learn joblib

In [62]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [63]:
df = pd.read_csv("clean_jobs_descriptions_combined (1).csv")

In [65]:
print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

Rows: 8785
Columns:
['Job Id', 'Job Title', 'Company', 'location', 'Job Description', 'Experience', 'Qualifications', 'Salary Range', 'Work Type', 'skills', 'clean_description', 'matched_skills', 'skill_categories']


In [68]:
print("clean_description:")
print(df["clean_description"].head(5))

print("\nextracted_skills:")
print(df["matched_skills"].head(5))

print("\ncategories:")
print(df["skill_categories"].head(5))

clean_description:
0    social media managers oversee an organizations...
1    frontend web developers design and implement u...
2    quality control managers establish and enforce...
3    wireless network engineers design implement an...
4    a conference manager coordinates and manages c...
Name: clean_description, dtype: object

extracted_skills:
0                                                  NaN
1    JavaScript (Programming), HTML (Programming), ...
2                                                  NaN
3                                                  NaN
4                                                  NaN
Name: matched_skills, dtype: object

categories:
0                             NaN
1    Programming, Web Development
2                             NaN
3                             NaN
4                             NaN
Name: skill_categories, dtype: object


In [71]:
labeled_df = df[
    df["matched_skills"].notna() &
    df["skill_categories"].notna()
].copy()

print("Total dataset rows:", len(df))
print("Rows with labeled skills:", len(labeled_df))

Total dataset rows: 8785
Rows with labeled skills: 1126


In [73]:
#create training data from extracted skills
training_records = []

for skills_text in labeled_df["matched_skills"].dropna():

    # Find: Skill Name (Category)
    matches = re.findall(
        r'([^,]+?)\s*\(([^)]+)\)',
        str(skills_text)
    )

    for skill, category in matches:

        skill = skill.strip()
        category = category.strip()

        if skill and category:

            training_records.append({
                "Text": skill,
                "Label": category
            })

training_data = pd.DataFrame(training_records)

print("Training examples:", len(training_data))

training_data.head(20)

Training examples: 3899


,Text,Label
0,JavaScript,Programming
1,HTML,Programming
2,CSS,Programming
3,HTML,Web Development
4,CSS,Web Development
5,React,Web Development
6,Angular,Web Development
7,Python,Programming
8,Java,Programming
9,JavaScript,Programming


In [74]:
training_data = training_data.drop_duplicates(
    subset=["Text", "Label"]
).reset_index(drop=True)

print("Unique training examples:", len(training_data))

training_data.head(20)

Unique training examples: 29


,Text,Label
0,JavaScript,Programming
1,HTML,Programming
2,CSS,Programming
3,HTML,Web Development
4,CSS,Web Development
5,React,Web Development
6,Angular,Web Development
7,Python,Programming
8,Java,Programming
9,SQL,Programming


In [75]:
print(
    training_data["Label"].value_counts()
)

Label
Programming         7
Web Development     4
Database            4
Data Analytics      3
DevOps              3
Cloud               3
Machine Learning    3
Big Data            2
Name: count, dtype: int64


In [76]:
important_skills = [
    "Python",
    "TensorFlow",
    "AWS",
    "SQL",
    "Power BI",
    "PyTorch"
]

for skill in important_skills:

    result = training_data[
        training_data["Text"].str.lower() == skill.lower()
    ]

    print(f"\n{skill}")

    if len(result) > 0:
        print(result)
    else:
        print("Not found in labeled training data")


Python
     Text        Label
7  Python  Programming

TensorFlow
          Text             Label
21  TensorFlow  Machine Learning

AWS
   Text  Label
17  AWS  Cloud

SQL
   Text        Label
9   SQL  Programming
10  SQL     Database

Power BI
        Text           Label
11  Power BI  Data Analytics

PyTorch
       Text             Label
22  PyTorch  Machine Learning


In [77]:
X = training_data["Text"]
y = training_data["Label"]

print("X samples:", len(X))
print("y samples:", len(y))

X samples: 29
y samples: 29


In [78]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,

)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 23
Testing samples: 6


In [79]:
#TF+IDF+Logistic Regression
model = Pipeline([

    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            sublinear_tf=True
        )
    ),

    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

print("TF-IDF + Logistic Regression pipeline created.")

TF-IDF + Logistic Regression pipeline created.


In [80]:
model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [82]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", round(accuracy, 2))

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

Accuracy: 0.17

Classification Report:

                  precision    recall  f1-score   support

  Data Analytics       0.00      0.00      0.00         1
        Database       0.00      0.00      0.00         2
Machine Learning       0.00      0.00      0.00         1
     Programming       0.20      0.50      0.29         2

        accuracy                           0.17         6
       macro avg       0.05      0.12      0.07         6
    weighted avg       0.07      0.17      0.10         6



In [83]:
test_skills = [
    "Python",
    "JavaScript",
    "HTML",
    "CSS",
    "SQL",
    "Power BI",
    "Tableau"
]

predictions = model.predict(test_skills)

test_results = pd.DataFrame({
    "Text": test_skills,
    "Predicted_Label": predictions
})

test_results

,Text,Predicted_Label
0,Python,Programming
1,JavaScript,Programming
2,HTML,Programming
3,CSS,Programming
4,SQL,Database
5,Power BI,Data Analytics
6,Tableau,Programming


In [84]:
skill_vocabulary = sorted(
    training_data["Text"]
    .dropna()
    .unique()
    .tolist(),
    key=len,
    reverse=True
)

print("Number of known skills:", len(skill_vocabulary))

print(skill_vocabulary[:50])

Number of known skills: 26
['Scikit-learn', 'JavaScript', 'TensorFlow', 'Kubernetes', 'Power BI', 'Angular', 'Tableau', 'MongoDB', 'PyTorch', 'Python', 'Hadoop', 'Docker', 'Oracle', 'React', 'Spark', 'Azure', 'Excel', 'MySQL', 'HTML', 'Java', 'CSS', 'SQL', 'Git', 'AWS', 'GCP', 'R']


In [85]:
#extract skills from description
def extract_candidate_skills(text, skill_vocabulary):

    if pd.isna(text):
        return []

    text = str(text).lower()

    found_skills = []

    for skill in skill_vocabulary:

        skill_lower = skill.lower().strip()

        # Escape special regex characters
        pattern = r"(?<!\w)" + re.escape(skill_lower) + r"(?!\w)"

        if re.search(pattern, text):

            found_skills.append(skill)

    return found_skills

In [86]:
def classify_clean_description(text):

    candidates = extract_candidate_skills(
        text,
        skill_vocabulary
    )

    results = []

    for skill in candidates:

        predicted_label = model.predict(
            [skill]
        )[0]

        results.append({
            "Text": skill,
            "Entity": skill,
            "Label": predicted_label
        })

    return results

In [88]:
sample_description = df["clean_description"].dropna().iloc[0]

print("CLEAN DESCRIPTION:\n")
print(sample_description)

print("\n\nEXTRACTED SKILLS:\n")

sample_result = classify_clean_description(
    sample_description
)

pd.DataFrame(sample_result)

CLEAN DESCRIPTION:

social media managers oversee an organizations social media presence they create and schedule content engage with followers and analyze social media metrics to drive brand awareness and engagement


EXTRACTED SKILLS:



""


In [89]:
results = []

for index, row in df.iterrows():

    description = row["clean_description"]

    extracted = classify_clean_description(
        description
    )

    for item in extracted:

        results.append({
            "Job Id": row["Job Id"],
            "Job Title": row["Job Title"],
            "clean_description": description,
            "Text": item["Text"],
            "Entity": item["Entity"],
            "Label": item["Label"]
        })

ml_skill_results = pd.DataFrame(results)

print(
    "Total extracted skill records:",
    len(ml_skill_results)
)

ml_skill_results.head(20)

Total extracted skill records: 191


,Job Id,Job Title,clean_description,Text,Entity,Label
0,3.038060e+15,Database Developer,sql database developers design implement and m...,SQL,SQL,Database
1,1.536530e+15,Java Developer,java backend developers specialize in building...,Java,Java,Programming
2,1.459740e+15,Front-End Engineer,javascript developers write code to create int...,JavaScript,JavaScript,Programming
3,1.724920e+15,Front-End Engineer,javascript developers write code to create int...,JavaScript,JavaScript,Programming
4,1.225540e+15,Java Developer,java software engineers develop and maintain s...,Java,Java,Programming
5,1.472330e+15,Web Designer,frontend web designers create the visual eleme...,JavaScript,JavaScript,Programming
6,1.472330e+15,Web Designer,frontend web designers create the visual eleme...,HTML,HTML,Programming
7,1.472330e+15,Web Designer,frontend web designers create the visual eleme...,CSS,CSS,Programming
8,1.662110e+15,Systems Engineer,as a cloud systems engineer you will be respon...,Azure,Azure,Cloud
9,1.662110e+15,Systems Engineer,as a cloud systems engineer you will be respon...,AWS,AWS,Cloud


In [90]:
python_results = ml_skill_results[
    ml_skill_results["Text"].str.lower() == "python"
]

python_results.head(20)

,Job Id,Job Title,clean_description,Text,Entity,Label


In [91]:
tensorflow_results = ml_skill_results[
    ml_skill_results["Text"].str.lower() == "tensorflow"
]

tensorflow_results.head(20)

,Job Id,Job Title,clean_description,Text,Entity,Label
